<a href="https://colab.research.google.com/github/doa-2026/machine-learning/blob/main/Abalone_per_proccessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Data

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
path="/content/drive/MyDrive/datanew.csv"
import pandas as pd
df=pd.read_csv(path)
df.head()
df

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


check type of columns

In [56]:
df.info()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sex             4177 non-null   object 
 1   length          4177 non-null   float64
 2   diameter        4177 non-null   float64
 3   height          4177 non-null   float64
 4   whole_weight    4177 non-null   float64
 5   shucked_weight  4177 non-null   float64
 6   viscera_weight  4177 non-null   float64
 7   shell_weight    4177 non-null   float64
 8   rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


,0
sex,object
length,float64
diameter,float64
height,float64
whole_weight,float64
shucked_weight,float64
viscera_weight,float64
shell_weight,float64
rings,int64


In [57]:
df.duplicated().sum()

np.int64(0)

Import Packages

In [58]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer




In [59]:
df["sex"].value_counts()

,count
sex,
M,1528
I,1342
F,1307


In [60]:
df.isna().sum()

,0
sex,0
length,0
diameter,0
height,0
whole_weight,0
shucked_weight,0
viscera_weight,0
shell_weight,0
rings,0


In [61]:
s=df.select_dtypes("object").columns
df["sex"].value_counts()

,count
sex,
M,1528
I,1342
F,1307


In [62]:
z=df.select_dtypes("number").columns
df.describe().round(7)

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,0.523992,0.407881,0.139516,0.828742,0.359368,0.180594,0.238831,9.933685
std,0.120093,0.099240,0.041827,0.490389,0.221963,0.109614,0.139203,3.224169
min,0.075000,0.055000,0.000000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.450000,0.350000,0.115000,0.441500,0.186000,0.093500,0.130000,8.000000
50%,0.545000,0.425000,0.140000,0.799500,0.336000,0.171000,0.234000,9.000000
75%,0.615000,0.480000,0.165000,1.153000,0.502000,0.253000,0.329000,11.000000
max,0.815000,0.650000,1.130000,2.825500,1.488000,0.760000,1.005000,29.000000


Validation Split

In [63]:
X=df.drop(columns=["rings"])
y=df["rings"]
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42)

Creat Column Transformer

In [67]:
col_num=X_train.select_dtypes("number").columns
scaler=StandardScaler()
num_imp=SimpleImputer(strategy="mean")
num_pip=make_pipeline(num_imp,scaler)
num_tup=("number",num_pip,col_num)
num_tup


('number',
 Pipeline(steps=[('simpleimputer', SimpleImputer()),
                 ('standardscaler', StandardScaler())]),
 Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
        'viscera_weight', 'shell_weight'],
       dtype='object'))

In [69]:
cat_col=X_train.select_dtypes("object").columns
cat_imp=SimpleImputer(strategy="constant",fill_value="Missing")
cat_hot=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
cat_pip=make_pipeline(cat_imp,cat_hot)
cat_tup=("cat",cat_pip,cat_col)
cat_tup

('cat',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='Missing', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['sex'], dtype='object'))

In [81]:
pre=ColumnTransformer([num_tup,cat_tup],verbose_feature_names_out=False)
pre_train=pre.fit_transform(X_train)
pre_test=pre.transform(X_test)
pre_train.info()
pre_test.dtypes

<class 'pandas.core.frame.DataFrame'>
Index: 3132 entries, 3823 to 860
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   length          3132 non-null   float64
 1   diameter        3132 non-null   float64
 2   height          3132 non-null   float64
 3   whole_weight    3132 non-null   float64
 4   shucked_weight  3132 non-null   float64
 5   viscera_weight  3132 non-null   float64
 6   shell_weight    3132 non-null   float64
 7   sex_F           3132 non-null   float64
 8   sex_I           3132 non-null   float64
 9   sex_M           3132 non-null   float64
dtypes: float64(10)
memory usage: 269.2 KB


,0
length,float64
diameter,float64
height,float64
whole_weight,float64
shucked_weight,float64
viscera_weight,float64
shell_weight,float64
sex_F,float64
sex_I,float64
sex_M,float64


In [83]:
pre_train.describe().round(2)
# all numerical feature at the same scale between  -1.7 , 5.48
# max of height 23.2 after scaling  but I  accepted

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
count,3132.00,3132.00,3132.00,3132.00,3132.00,3132.00,3132.00,3132.00,3132.00,3132.00
mean,-0.00,0.00,0.00,-0.00,0.00,-0.00,0.00,0.32,0.32,0.37
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.47,0.47,0.48
min,-3.78,-3.59,-3.28,-1.69,-1.62,-1.66,-1.71,0.00,0.00,0.00
25%,-0.64,-0.60,-0.59,-0.79,-0.79,-0.81,-0.79,0.00,0.00,0.00
50%,0.16,0.16,0.12,-0.06,-0.10,-0.09,-0.04,0.00,0.00,0.00
75%,0.75,0.72,0.58,0.66,0.65,0.67,0.64,1.00,1.00,1.00
max,2.43,2.44,23.21,4.05,5.05,5.29,5.48,1.00,1.00,1.00
